In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-hc2ff5eu/unsloth_fc48a0d55c624390a683be1bd3b50280
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-hc2ff5eu/unsloth_fc48a0d55c624390a683be1bd3b50280
  Resolved https://github.com/unslothai/unsloth.git to commit b80239946126dc2e2dc45b4f5dede781e9ec3391
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [3]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024 # Unsloth handles RoPE scaling automatically

# 1. Load the base model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True, # 4-bit quantization for VRAM efficiency
)

# 2. Apply the LoRA configuration
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0, # Unsloth optimizes dropout to 0
    bias="none",    # Unsloth optimizes bias to "none"
    use_gradient_checkpointing=False,
    random_state=3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.5: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Unsloth 2026.9.5 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


In [4]:
from datasets import load_dataset

# We use a secure Parquet port of the exact same dataset
dataset = load_dataset("OpenRL/daily_dialog")

def format_daily_dialog(example):
    messages = []
    # The Parquet version maintains the 'dialog' key as a list of conversation turns
    for i, turn in enumerate(example["dialog"]):
        # Even indices (0, 2, 4...) are the user, odd indices (1, 3...) are the assistant
        role = "user" if i % 2 == 0 else "assistant"
        messages.append({"role": role, "content": turn.strip()})

    # Safely convert the dictionaries into Qwen's training format
    example["text"] = tokenizer.apply_chat_template(messages, tokenize=False)
    return example

# Map the formatting function to the training split
train_dataset = dataset["train"].map(format_daily_dialog)
val_dataset = dataset["validation"].map(format_daily_dialog)

In [5]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,          # moved here
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=SFTConfig(
        per_device_train_batch_size=16,
        gradient_accumulation_steps=1,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        output_dir="outputs",
        seed=3407,
        eval_strategy="epoch",
        per_device_eval_batch_size=4,
        # eval_dataset removed from here
    ),
)
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
# Start training
trainer_stats = trainer.train()



Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/11118 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map (num_proc=2):   0%|          | 0/11118 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]

...

Epoch,Training Loss,Validation Loss
1,2.192409,2.074985
2,1.824002,2.030570
3,1.529956,2.074077


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.


Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-2000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-2085/tokenizer_config.json.


RuntimeError: on_train_begin must be called before on_evaluate

In [6]:
# 1. Enable Unsloth's optimized 2x faster inference mode
FastLanguageModel.for_inference(model)

# 2. Write a test message (something similar to a DailyDialog conversation)
messages = [
    {"role": "user", "content": "Hey! What's up"}
]

# 3. Apply the exact same ChatML formatting we used during training
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True, # Crucial: Tells the model it is the assistant's turn to speak
    return_tensors="pt",
).to("cuda") # Move the inputs to the T4 GPU

# 4. Generate the response
# You can tweak temperature and max_new_tokens to change the style/length
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=128,
    use_cache=True,
    temperature=0.7,
    do_sample=True
)

# 5. Decode the output
# We slice the output array [inputs.shape[1]:] so it only prints the NEW text, not your prompt
response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

print("User: ", messages[0]["content"])
print("Model:", response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User:  Hey! What's up
Model: Nothing much . How about you ?


In [7]:
import math

eval_logs = [log for log in trainer.state.log_history if "eval_loss" in log]
eval_loss = eval_logs[-1]["eval_loss"]

print(f"Final epoch eval loss: {eval_loss:.4f}")
print(f"Perplexity: {math.exp(eval_loss):.2f}")

Final epoch eval loss: 2.0741
Perplexity: 7.96


In [8]:
# Replace "your-username" with your actual Hugging Face username!
repo_name = "TanishkDhope/tetherchat-smart-replies"

# Unsloth merges the weights and uploads them to HF in one step
model.push_to_hub_merged(
    repo_name,
    tokenizer,
    save_method="merged_16bit",
)

print(f"Success! Your model is now live at: https://huggingface.co/{repo_name}")

config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.
Unsloth: Restored added_tokens_decoder metadata in TanishkDhope/tetherchat-smart-replies/tokenizer_config.json.
No files have been modified since last commit. Skipping to prevent empty commit.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...replies/model.safetensors:   3%|2         | 27.9MB /  988MB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:26<00:00, 26.18s/it]


Unsloth: Merge process complete. Saved to `/content/TanishkDhope/tetherchat-smart-replies`
Success! Your model is now live at: https://huggingface.co/TanishkDhope/tetherchat-smart-replies


In [9]:
test_messages = [
    "Hey! Want to grab coffee?",
    "I finished the report, sending it now.",
    "Can we push the meeting to 3pm?",
    "Ugh, traffic is insane today.",
]
for msg in test_messages:
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": msg}],
        tokenize=True, add_generation_prompt=True, return_tensors="pt",
    ).to("cuda")
    output = model.generate(input_ids=inputs, max_new_tokens=40, temperature=0.7, do_sample=True)
    print(f"User: {msg}\nModel: {tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)}\n")

Both `max_new_tokens` (=40) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User: Hey! Want to grab coffee?
Model: Sure ! What time do you want me there ?



Both `max_new_tokens` (=40) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User: I finished the report, sending it now.
Model: Thank you for your help .



Both `max_new_tokens` (=40) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User: Can we push the meeting to 3pm?
Model: Sorry , but I have to go to the library .

User: Ugh, traffic is insane today.
Model: I wish there were no traffic jams .



## Export GGUF for CPU serving

The `smart-replies/` FastAPI service runs the model on CPU with llama.cpp, which needs a GGUF file. Run this in a **fresh runtime** (the merged fp16 repo is public, so no login is needed to read it — only to push). It adds `tetherchat-smart-replies.Q8_0.gguf` and `tetherchat-smart-replies.Q4_K_M.gguf` to the same Hub repo; the service defaults to Q8_0 (`MODEL_FILE` env var).

In [ ]:
from unsloth import FastLanguageModel

repo_name = "TanishkDhope/tetherchat-smart-replies"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=repo_name,
    max_seq_length=1024,
    load_in_4bit=False,   # load the merged fp16 weights, not a 4-bit view
)
model.push_to_hub_gguf(repo_name, tokenizer, quantization_method=["q8_0", "q4_k_m"])
print(f"Check the filenames at https://huggingface.co/{repo_name}/tree/main and set MODEL_FILE accordingly.")